# 🌿 TerraMind Afrique — Fine-Tuning du Modèle de Diagnostic (`terramind_v1`)
**Responsable : Oscar (IA & Données)** · **PAS Challenge 2026 (SALTIS)**

Ce notebook exécute toute la chaîne d'entraînement sur **Google Colab (GPU T4)** :
1. Clone le dépôt `Terramind` et installe les dépendances.
2. Télécharge directement sur les serveurs rapides de Colab les deux jeux bruts (**TOM2024 Burkina Faso** + **CCMT Ghana Raw Data**).
3. Construit la base nettoyée `terramind_v1` (35 448 images, exclusion SHA-256 des doublons et images illisibles) et le découpage stratifié `train / val / test` (80 / 10 / 10).
4. Fine-tune le modèle pré-entraîné sur les maladies des plantes **`linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification`** (Hugging Face) sur `terramind_v1`.
5. Calibre le `seuil_incertain`, génère `ml/eval/metrics.json` et télécharge automatiquement les poids (`terramind_v1_model.pt`) et les métriques sur ton PC.

> **Avant de lancer :** Dans Colab, vérifie que tu es bien en **Exécution → Modifier le type d'exécution → GPU T4**.

In [ ]:
# 1. Vérification du GPU et clonage du dépôt
!nvidia-smi
!git clone -b ml/dataset-merge https://github.com/Oscar-AS/Terramind.git
%cd Terramind
!pip install -q datasets transformers accelerate pillow torchvision kagglehub

In [ ]:
# 2. Téléchargement rapide de TOM2024 (Hugging Face) et CCMT Raw Data (KaggleHub)
from pathlib import Path
import shutil
from datasets import load_dataset
import kagglehub

raw_dir = Path('data_local/raw')
raw_dir.mkdir(parents=True, exist_ok=True)

print('Téléchargement de TOM2024 (raw)...')
ds = load_dataset('Project-AgML/TOM2024_disease_classification', 'raw', split='train')
ds.save_to_disk('data_local/raw/tom2024_raw')

print('Téléchargement de CCMT Raw Data...')
ccmt_path = Path(kagglehub.dataset_download('irakozekelly/crop-pest-and-disease-dataset'))
target_ccmt = Path('data_local/raw/ccmt_raw/CCMT Dataset')
target_ccmt.parent.mkdir(parents=True, exist_ok=True)
for sub in ccmt_path.rglob('Raw Data'):
    if sub.is_dir():
        shutil.copytree(sub, target_ccmt, dirs_exist_ok=True)
        break
print('Jeux bruts prêts !')

In [ ]:
# 3. Construction de terramind_v1 (D12 & D13) et découpage stratifié train/val/test (80/10/10)
!python ml/training/build_merged_dataset.py
!python ml/training/split_dataset.py

In [ ]:
# 4. Fine-tuning du modèle Hugging Face pré-entraîné PlantVillage sur terramind_v1
# (Option --crops tomato,onion,maize disponible si on veut restreindre aux cultures maraîchères/céréalières)
!python ml/training/train_classifier.py \
    --backbone linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification \
    --epochs-head 2 \
    --epochs-finetune 6 \
    --batch-size 64 \
    --lr 1e-4 \
    --leaf-only

In [ ]:
# 5. Affichage du rapport d'évaluation et téléchargement des fichiers vers ton PC
import json
from pathlib import Path

metrics = json.loads(Path('ml/eval/metrics.json').read_text(encoding='utf-8'))
print('=== RÉSULTATS SUR LE JEU DE TEST ===')
print('Exactitude globale :', metrics['test_metrics']['exactitude_globale'])
print('Seuil incertain calibré :', metrics['seuil_incertain'])
print('Exactitude hors incertain :', metrics['test_metrics']['exactitude_hors_incertain'])
print('Taux incertain :', metrics['test_metrics']['taux_incertain'])
print('Par source :', metrics['test_metrics']['par_source'])
print('Par culture :', metrics['test_metrics']['par_culture'])

try:
    from google.colab import files
    files.download('ml/eval/metrics.json')
    files.download('modeles/diagnostic/classes.json')
    files.download('modeles/diagnostic/terramind_v1_model.pt')
except ImportError:
    print('Exécution hors Colab : fichiers disponibles dans modeles/diagnostic/ et ml/eval/.')